# BioLaya Sprint 1 — Google Colab + Google Drive

**Canonical Sprint 1 Colab notebook.** Download this `.ipynb` from the BioLaya ZIP/GitHub, put it in Google Drive, and open it with Google Colab.

It installs BioLaya directly from GitHub with `pip`; no repository clone is required in `/content`.


## 0. Runtime

For GPU evaluation/training choose **Runtime → Change runtime type → GPU**. CPU mode also works for smoke tests.


In [ ]:
DEVICE = "auto"  # "auto", "cpu", or "cuda"
DRIVE_ROOT = "/content/drive/MyDrive/BioLaya"
GITHUB_REPO = "https://github.com/Gabriel382/BioLaya.git"
print(DEVICE, DRIVE_ROOT, GITHUB_REPO)


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Install BioLaya directly from GitHub

Re-running this cell after a GitHub update installs the newest code from `main`.


In [ ]:
%pip install -q --upgrade "biolaya[train,colab] @ git+https://github.com/Gabriel382/BioLaya.git"


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
import fsspec, gcsfs
print("fsspec:", fsspec.__version__)
print("gcsfs:", gcsfs.__version__)


## 3. Verify device and initialize persistent Drive storage


In [ ]:
import json
from biolaya.cloud.env import environment_report, resolve_device
from biolaya.cloud.drive import initialize_drive

print(json.dumps(environment_report(DEVICE), indent=2))
RESOLVED_DEVICE = resolve_device(DEVICE)
layout = initialize_drive(DRIVE_ROOT)
print('resolved device:', RESOLVED_DEVICE)
print('drive root:', layout.root)


## 4. Fast structural smoke test

No model download. Validates package, device contract, schemas, and Drive checkpoint backup/restore.


In [ ]:
!biolaya-smoke --device {DEVICE} --skip-model


## 5. Prepare Sprint 1 data on the Colab VM

Datasets stay under `/content`; Drive is reserved for persistent artifacts.


In [ ]:
from pathlib import Path
DATA_ROOT = Path('/content/biolaya_data/processed')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
!biolaya-data bionli nli4ct --root {DATA_ROOT}


## 6. Real Laya inference smoke test

Downloads the upstream Laya model on first use and runs one real typed-decision inference.


In [ ]:
!biolaya-smoke --device {DEVICE}


## 7. Base-Laya benchmark smoke test


In [ ]:
RESULT_LOCAL = '/content/biolaya_results/sprint1/laya_base_bionli_smoke'
!biolaya-eval {DATA_ROOT}/bionli/test.jsonl --model english --device {DEVICE} --max-examples 50 --output-dir {RESULT_LOCAL}


## 8. Persist result to Google Drive


In [ ]:
from pathlib import Path
import shutil

src = Path(RESULT_LOCAL)
dst = layout.results / 'sprint1' / 'laya_base_bionli_smoke'
if dst.exists():
    shutil.rmtree(dst)
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, dst)
print('saved:', dst)


## Sprint 1 next step

Once the smoke test passes, prepare all five datasets with:

```python
!biolaya-data --all --root /content/biolaya_data/processed
```

Future sprint notebooks follow the same pattern: install from GitHub → work under `/content` → persist checkpoints/results to Drive → resume after a new session.
